<a href="https://colab.research.google.com/github/ateachment/MoodleSearch/blob/main/moodleSearch_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Lemmatisierung deutscher Sprache: 
https://nickyreinert.de/blog/2020/12/09/einfuehrung-in-stemming-und-lemmatisierung-deutscher-texte-mit-python/

Installation von HanoverTagger (wird :

In [46]:
# pip install HanTa 

In [47]:
import requests
import bs4
import nltk
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')

from HanTa import HanoverTagger as ht
hannover = ht.HanoverTagger('morphmodel_ger.pgz')

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity # We will use this later to decide how similar two sentences are

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\838235\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\838235\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\838235\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [48]:
stop = nltk.corpus.stopwords.words('german')
# Add a few more stop words we would like to remove here
stop.append('daher')
stop.append('vieler')
stop.append('vielen')
stop.append('usw')
stop.append('bzw')
stop.append('etc')
stop.append('d.h.')
stop.append('u.a')
stop.append('z.b')
stop.append('--')
stop.append('-')
stop.append('``')
stop.append("''")
stop

['aber',
 'alle',
 'allem',
 'allen',
 'aller',
 'alles',
 'als',
 'also',
 'am',
 'an',
 'ander',
 'andere',
 'anderem',
 'anderen',
 'anderer',
 'anderes',
 'anderm',
 'andern',
 'anderr',
 'anders',
 'auch',
 'auf',
 'aus',
 'bei',
 'bin',
 'bis',
 'bist',
 'da',
 'damit',
 'dann',
 'der',
 'den',
 'des',
 'dem',
 'die',
 'das',
 'dass',
 'daß',
 'derselbe',
 'derselben',
 'denselben',
 'desselben',
 'demselben',
 'dieselbe',
 'dieselben',
 'dasselbe',
 'dazu',
 'dein',
 'deine',
 'deinem',
 'deinen',
 'deiner',
 'deines',
 'denn',
 'derer',
 'dessen',
 'dich',
 'dir',
 'du',
 'dies',
 'diese',
 'diesem',
 'diesen',
 'dieser',
 'dieses',
 'doch',
 'dort',
 'durch',
 'ein',
 'eine',
 'einem',
 'einen',
 'einer',
 'eines',
 'einig',
 'einige',
 'einigem',
 'einigen',
 'einiger',
 'einiges',
 'einmal',
 'er',
 'ihn',
 'ihm',
 'es',
 'etwas',
 'euer',
 'eure',
 'eurem',
 'euren',
 'eurer',
 'eures',
 'für',
 'gegen',
 'gewesen',
 'hab',
 'habe',
 'haben',
 'hat',
 'hatte',
 'hatten',
 '

# Web Crawling

In [49]:
session = requests.Session()
session.headers['User-Agent'] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:130.0) Gecko/20100101 Firefox/130.0"

base_url = 'https://moodle.eick-at.de/course/view.php?id=21'  # URL of the Moodle Course "test"
r = session.get(base_url, allow_redirects=True, timeout=10)   # Make a GET request to the URL and store the response in r
r


<Response [200]>

In [ ]:
#`r.text` contains the raw HTML returned when we made our GET request earlier. 
#`'html5lib'` tells BeautifulSoup that it is reading HTML information. 
soup = bs4.BeautifulSoup(r.text,'html5lib')
# script und style entfernen
for tag in soup(["script", "style"]):
    tag.decompose()
#print(soup.prettify()) # This will print the HTML in a more readable format. You can also just write `soup` to get the same result, but it is not as well formatted.



<!DOCTYPE html>
<html dir="ltr" lang="en" xml:lang="en">
 <head>
  <title>
   Course: test | eick-at moodle
  </title>
  <link href="https://moodle.eick-at.de/pluginfile.php/1/core_admin/favicon/64x64/1765395551/favicon.ico" rel="shortcut icon"/>
  <meta content="text/html; charset=utf-8" http-equiv="Content-Type"/>
  <meta content="moodle, Course: test | eick-at moodle" name="keywords"/>
  <link href="https://moodle.eick-at.de/theme/yui_combo.php?rollup/3.18.1/yui-moodlesimple-min.css" rel="stylesheet" type="text/css"/>
  <link href="https://moodle.eick-at.de/theme/styles.php/boost/1765395551_1/all" rel="stylesheet" type="text/css"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
 </head>
 <body class="format-topics limitedwidth path-course path-course-view gecko dir-ltr lang-en yui-skin-sam yui3-skin-sam moodle-eick-at-de pagelayout-course course-21 context-1151 category-1 theme uses-drawers drawer-open-index" id="page-course-view-topics">
  <div aria-live=

In [51]:
rawTexts = []       # list of all raw texts
shortTexts = []     # list of all short texts (section name + summarytext or linktext)
shortText = ["Das ist ein kleiner Test u.a. zum moodle crawlen - keyword spezial"]       # temporary variable to store the current short text (section name + summarytext or linktext)


def scrapePage(link,linkText,shortText):
  short_txt = ""
  head = requests.head(link, allow_redirects=True)
  header = head.headers
  content_type = header.get('content-type')
  if content_type == "text/html; charset=utf-8":  # follow only html content
    r = session.get(link)                         # Seiten aufrufen
    #print(content_type, link)
    if r.ok:
      soup = bs4.BeautifulSoup(r.text,'html.parser')
      markUpPageContent=soup.find("div", {"id": "page-content"})
      if markUpPageContent is not None:  # link to source code etc.
        #for tag in markUpPageContent.select('form, navitem, gradingsummary'):      # get rid of forms
        #  tag.decompose()
        for tag in soup.select('span.nolink'):      # get rid of LaTeX formules
          tag.decompose()
        for tag in soup.select('div.mediaplugin'):  # get rid of Videos
          tag.decompose()
        for tag in soup.select('iframe'):           # get rid of iframes
          tag.decompose()
        for tag in soup.select('div.footer-content-popover'):  # get rid of Contents of question mark button
          tag.decompose()
        for tag in soup.select('div.drawer'):  # get rid of Contents of question mark button
          tag.decompose()

        for br in soup.select("br"):
            br.replace_with(" ")

        #print(markUpPageContent)
        paragraphs = markUpPageContent.find_all(['p','h1','h2','h3','h4','h5','h6'])   # Absätze
        if(paragraphs is not None):
          txt = ""
          short_txt = ""
          for paragraph in paragraphs:
            txt += paragraph.get_text() + ' '
            if(len(short_txt) < 50 ):
              short_txt += paragraph.get_text()
        rawTexts[-1] += txt                                # add txt of page to last element of rawTexts
        shortTexts[-1] += [short_txt]         # add short_txt to last element of shortTexts
        '''
        print("RAW")
        print(rawTexts[-1])

        print("SHORT")
        print(shortTexts[-1])
        
        print("TXT2")
        print(txt)
        print("RAW")
        print(rawTexts)
        print("SHORT")
        print(shortTexts)
        '''

In [52]:
def cutText(txt): # gets list of words an extract the first 18 of them
  wordlength = 18
  wordlist = txt.split(' ')
  if len(wordlist) > wordlength:
    return ' '.join(wordlist[:wordlength]) + " ..."
  else:
    return txt

In [ ]:
link = base_url
links = []          # list of all links


for section in soup.find_all('li',{'class':'section'}):            #    ('h3',{'class':'sectionname'}):  # iterate sections
    #print("SECTION=",section)
    section_id = section.attrs['id']
    print("SECTION_ID=",section_id)
    sectionname = section.select_one(".sectionname").get_text(strip=True, separator=" ")
    print("SECTIONNAME=",sectionname)

    link2 = section.select_one(".sectionname a")["href"]         # find link of section
    
    print("LINK2=",link2)
    links.append(link2)                                   # append link to list links
    #print(sectionname)
    rawTexts.append(sectionname)                          # append text to list rawTexts

    #print(link2, shortTexts[-1])
    summarytext = section.find('div', attrs={'class':'summarytext'})
    if (summarytext is not None):
        summarytext = summarytext.get_text(strip=True, separator=" ")
        shortTexts.append(shortText + [sectionname] + [cutText(summarytext)])          # append shortText
    else:
        shortTexts.append(shortText + [sectionname])          # append shortText


    for markUpLink in section.find_all('a', attrs={'class':'aalink'}):
        #print(markUpLink)
        link3 = markUpLink.attrs['href']
        print("LINK3=",link3)
        if(link3 is not None):
            markUpName = markUpLink.span
            if not("Link/URL" in markUpName.get_text()):
                for s in markUpName.select('span'):
                    s.extract()
                    linkText = markUpName.get_text()
                    print("NOCHMAL LINK3=",link3)
                    #print(linkText)
                    links.append(link3)
                    rawTexts.append(linkText)
                    shortTexts.append(shortText + [sectionname] + [linkText])
                    scrapePage(link3,linkText,shortText + [sectionname] + [linkText])
                    #exit();
         

SECTION_ID= section-0
SECTIONNAME= General
LINK2= https://moodle.eick-at.de/course/section.php?id=184
LINK3= https://moodle.eick-at.de/mod/forum/view.php?id=1034
LINK3= https://moodle.eick-at.de/mod/quiz/view.php?id=1035
SECTION_ID= section-1
SECTIONNAME= New section
LINK2= https://moodle.eick-at.de/course/section.php?id=185
LINK3= https://moodle.eick-at.de/mod/page/view.php?id=1229
LINK3= https://moodle.eick-at.de/mod/page/view.php?id=1230


In [ ]:
def processText(txt):
  tokenized = nltk.tokenize.word_tokenize(txt)
  #print(tokenized)
  hannovered = [hannover.analyze(word)[0] for word in tokenized]
  #print(hannovered)
  processed = [w.lower() for w in hannovered if w not in stop]
  #print(processed)
  return processed


In [ ]:
data = []                                      
for header in headers:                          # iterate sections
  # print(header)
  link = header.find('a').attrs['href']         # extract link of section
  txt = header.get_text()                       # extract text of section
  processed = processText(txt)                  # nlp of section text
  entry = [link, processed]                     # list entry with link and processed text
  print(entry)
  data.append(entry)                            # append entry in list data (nested lists)

  listTags = header.find_next_sibling('ul')     # pages, tasks, links ..
  for li in listTags:
    #print(li)
    link = li.find('a')
    txt2 = link.get_text().replace("Page","").replace("URL","").replace("Assignment","")
    if not("URL" in link.get_text()):
      link = link.attrs['href'] 
      
      r = requests.get(link)                    # Seiten aufrufen
      soup2 = bs4.BeautifulSoup(r.text,'html5lib')
      header2 = soup2.find('h2')                # Überschrift
      #print(header2)
      paragraphs = header2.find_all_next('p')   # Absätze
      txt = ""
      for paragraph in paragraphs:
        txt += paragraph.get_text()
      txt2 += txt
      #print(link,txt2)
      processed =processText(txt2)
      entry = [link, processed]
      print(entry)
      data.append(entry)



NameError: name 'headers' is not defined

In [ ]:
import matplotlib.pyplot as plt
def plot_hist(data):
    entry_lengths = [len(entry[1]) for entry in data]
    fig = plt.figure(figsize=(6, 6)) 
    plt.xlabel('Eintraglänge')
    plt.ylabel('Anzahl der Einträge')
    plt.hist(entry_lengths, bins=20)
    plt.show()
    return entry_lengths
entry_lengths = plot_hist(data)

Bag of words

In [ ]:
# calculate frequency of words
def map_book(hash_map, tokens):
    if tokens is not None:
        for word in tokens:
            # Word Exist?
            if word in hash_map:
                hash_map[word] = hash_map[word] + 1
            else:
                hash_map[word] = 1

        return hash_map
    else:
        return None
        
def make_hash_map(data):  
    hash_map = {}
    for entry in data:
        hash_map = map_book(hash_map, entry[1])
    return hash_map


# define a function frequent_vocab with the following input: word_freq and max_features
def frequent_vocab(word_freq, max_features): 
    counter = 0  #initialize counter with the value zero
    vocab = []   # create an empty list called vocab
    # list words in the dictionary in descending order of frequency
    for key, value in sorted(word_freq.items(), key=lambda item: (item[1], item[0]), reverse=True): 
       #loop function to get the top (max_features) number of words
        if counter<max_features: 
            vocab.append(key)
            counter+=1
        else: break
    return vocab

In [ ]:
hash_map = make_hash_map(data) #create hash map (words and frequency) from tokenized dataset

vocab=frequent_vocab(hash_map, 1000)  # adjust second Parameter
print(hash_map)
print(vocab)

In [ ]:
# define a function bagofwords with the following input: page and words
def bagofwords(data, vocab):
    # frequency word count
    bag = np.zeros(len(vocab)) #create a NumPy array made up of zeroes with size len(words)
    # loop through data and add value of 1 when token is present in the tweet
    for sw in data:
        for i,word in enumerate(vocab):
            if word == sw: 
                bag[i] += 1
                
    return np.array(bag) # return the bag of word for one page

In [ ]:
test = ['künstlich', 'intelligenz', 'maschinell']
bagofwords(test, vocab)

In [ ]:
# set up a NumPy array with the specified dimension to contain the bag of words
n_words = len(vocab)
n_docs = len(data)
bag_o = np.zeros([n_docs,n_words])
# use loop function to add new row for each data of page. 
for ii in range(n_docs): 
    #call out the previous function 'bagofwords'. see the inputs: sentence and words
    bag_o[ii,:] = bagofwords(data[ii][1], vocab) 

In [ ]:
bag_o.shape

Inverse document frequency

In [ ]:
#initialize 2 variables representing the number of pages (numdocs) and the number of tokens/words (numwords)
numdocs, numwords = np.shape(bag_o)

#Changing into the tfidf formula as above
N = numdocs
term_frequency = np.empty(numwords)

#Count the number of documents the word appears in.
for word in range(numwords):
    term_frequency[word]=np.sum(bag_o[:,word]>0) 
print(term_frequency)
idf = np.log(N/term_frequency)
print(idf)

In [ ]:
#initializs tfidf array
tfidf = np.empty([numdocs, numwords])

#loop through the pages, multiply term frequency (represented by bag of words) with idf
for doc in range(numdocs):
    tfidf[doc, :]=bag_o[doc, :]*idf

In [ ]:
tfidf.shape

In [ ]:
print (tfidf)

This data can be saved so that it does not have to be determined each time with web crawling and NLP (could be done as a task once a day).

In [ ]:
filename = "tfidf.npy"   # file extension has to be "npy"
np.save(filename,tfidf)  # numpy provides file functions

The data can be loaded by this:

In [ ]:
tfidf=np.load(filename)

In [ ]:
# search string
search='Anaconda installieren maschinell maschinell künstlich'

processed = processText(search)
print(processed)
search_vector = bagofwords(processed, vocab)
print(search_vector)

#calculate tfidf 
term_frequency = np.empty(numwords)

#Count the number of documents the search word appears in.
for word in range(numwords):
    term_frequency[word]=np.sum(search_vector[word]>0) 
print(term_frequency)

#initializs tfidf array
search_tfidf = np.empty([numwords])

#multiply term frequency (represented by bag of words) with idf
search_tfidf = term_frequency * idf

print(search_tfidf)

In [ ]:
#comparision with search vector without tfidf
comparisons = cosine_similarity(tfidf, search_vector.reshape(1,-1))
print(comparisons)
#comparision with tfdif search vector => better results
comparisons = cosine_similarity(tfidf, search_tfidf.reshape(1,-1))
print(comparisons)

In [ ]:
# best result
print(data[comparisons.argmax()][0])